In [171]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

sns.set_theme(style="whitegrid")

In [172]:
df = pd.read_csv("../data/raw/data.csv",sep=";")
df

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,1,1,6,9773,1,1,125.0,1,1,1,...,0,6,8,5,12.666667,0,15.5,2.8,-4.06,Graduate
4420,1,1,2,9773,1,1,120.0,105,1,1,...,0,6,6,2,11.000000,0,11.1,0.6,2.02,Dropout
4421,1,1,1,9500,1,1,154.0,1,37,37,...,0,8,9,1,13.500000,0,13.9,-0.3,0.79,Dropout
4422,1,1,1,9147,1,1,180.0,1,37,37,...,0,5,6,5,12.000000,0,9.4,-0.8,-3.12,Graduate


In [173]:
df.shape

(4424, 37)

In [174]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                          4424

In [175]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Marital status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
Application mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
Application order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
Course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
Daytime/evening attendance\t,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
Previous qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
Previous qualification (grade),4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
Nacionality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
Mother's qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
Father's qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


In [176]:
df.columns

Index(['Marital status', 'Application mode', 'Application order', 'Course',
       'Daytime/evening attendance\t', 'Previous qualification',
       'Previous qualification (grade)', 'Nacionality',
       'Mother's qualification', 'Father's qualification',
       'Mother's occupation', 'Father's occupation', 'Admission grade',
       'Displaced', 'Educational special needs', 'Debtor',
       'Tuition fees up to date', 'Gender', 'Scholarship holder',
       'Age at enrollment', 'International',
       'Curricular units 1st sem (credited)',
       'Curricular units 1st sem (enrolled)',
       'Curricular units 1st sem (evaluations)',
       'Curricular units 1st sem (approved)',
       'Curricular units 1st sem (grade)',
       'Curricular units 1st sem (without evaluations)',
       'Curricular units 2nd sem (credited)',
       'Curricular units 2nd sem (enrolled)',
       'Curricular units 2nd sem (evaluations)',
       'Curricular units 2nd sem (approved)',
       'Curricular units 2nd

In [177]:
df_clean = df.copy()

In [178]:
df_clean.columns = (df_clean.columns
                    .str.strip()
                    .str.lower()
                    .str.replace(" ", "_")
                    .str.replace("/", "_")
                    .str.replace("'", "")
                    .str.replace("(", "", regex=False)
                    .str.replace(")", "", regex=False))

In [179]:
df_clean = df_clean.rename(columns={"nacionality": "nationality"})

In [180]:
df_clean.columns

Index(['marital_status', 'application_mode', 'application_order', 'course',
       'daytime_evening_attendance', 'previous_qualification',
       'previous_qualification_grade', 'nationality', 'mothers_qualification',
       'fathers_qualification', 'mothers_occupation', 'fathers_occupation',
       'admission_grade', 'displaced', 'educational_special_needs', 'debtor',
       'tuition_fees_up_to_date', 'gender', 'scholarship_holder',
       'age_at_enrollment', 'international',
       'curricular_units_1st_sem_credited',
       'curricular_units_1st_sem_enrolled',
       'curricular_units_1st_sem_evaluations',
       'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade',
       'curricular_units_1st_sem_without_evaluations',
       'curricular_units_2nd_sem_credited',
       'curricular_units_2nd_sem_enrolled',
       'curricular_units_2nd_sem_evaluations',
       'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade',
       'curricular_units_2nd_sem_wit

In [181]:
df_clean.duplicated().sum()

np.int64(0)

In [182]:
df_clean.isna().sum()

marital_status                                  0
application_mode                                0
application_order                               0
course                                          0
daytime_evening_attendance                      0
previous_qualification                          0
previous_qualification_grade                    0
nationality                                     0
mothers_qualification                           0
fathers_qualification                           0
mothers_occupation                              0
fathers_occupation                              0
admission_grade                                 0
displaced                                       0
educational_special_needs                       0
debtor                                          0
tuition_fees_up_to_date                         0
gender                                          0
scholarship_holder                              0
age_at_enrollment                               0


In [183]:
df_clean.dtypes

marital_status                                    int64
application_mode                                  int64
application_order                                 int64
course                                            int64
daytime_evening_attendance                        int64
previous_qualification                            int64
previous_qualification_grade                    float64
nationality                                       int64
mothers_qualification                             int64
fathers_qualification                             int64
mothers_occupation                                int64
fathers_occupation                                int64
admission_grade                                 float64
displaced                                         int64
educational_special_needs                         int64
debtor                                            int64
tuition_fees_up_to_date                           int64
gender                                          

In [184]:
df_clean.nunique()

marital_status                                    6
application_mode                                 18
application_order                                 8
course                                           17
daytime_evening_attendance                        2
previous_qualification                           17
previous_qualification_grade                    101
nationality                                      21
mothers_qualification                            29
fathers_qualification                            34
mothers_occupation                               32
fathers_occupation                               46
admission_grade                                 620
displaced                                         2
educational_special_needs                         2
debtor                                            2
tuition_fees_up_to_date                           2
gender                                            2
scholarship_holder                                2
age_at_enrol

In [185]:
df_clean.describe().T

,count,mean,std,min,25%,50%,75%,max
marital_status,4424.0,1.178571,0.605747,1.00,1.00,1.000000,1.000000,6.000000
application_mode,4424.0,18.669078,17.484682,1.00,1.00,17.000000,39.000000,57.000000
application_order,4424.0,1.727848,1.313793,0.00,1.00,1.000000,2.000000,9.000000
course,4424.0,8856.642631,2063.566416,33.00,9085.00,9238.000000,9556.000000,9991.000000
daytime_evening_attendance,4424.0,0.890823,0.311897,0.00,1.00,1.000000,1.000000,1.000000
previous_qualification,4424.0,4.577758,10.216592,1.00,1.00,1.000000,1.000000,43.000000
previous_qualification_grade,4424.0,132.613314,13.188332,95.00,125.00,133.100000,140.000000,190.000000
nationality,4424.0,1.873192,6.914514,1.00,1.00,1.000000,1.000000,109.000000
mothers_qualification,4424.0,19.561935,15.603186,1.00,2.00,19.000000,37.000000,44.000000
fathers_qualification,4424.0,22.275316,15.343108,1.00,3.00,19.000000,37.000000,44.000000


In [186]:
(df_clean.select_dtypes(include="number") == 0).sum()

marital_status                                     0
application_mode                                   0
application_order                                  1
course                                             0
daytime_evening_attendance                       483
previous_qualification                             0
previous_qualification_grade                       0
nationality                                        0
mothers_qualification                              0
fathers_qualification                              0
mothers_occupation                               144
fathers_occupation                               128
admission_grade                                    0
displaced                                       1998
educational_special_needs                       4373
debtor                                          3921
tuition_fees_up_to_date                          528
gender                                          2868
scholarship_holder                            

In [187]:
df_clean["target"] = df_clean["target"].str.strip()
df_clean["target"].value_counts()

target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

In [188]:
df_clean["target"].value_counts(normalize=True) * 100

target
Graduate    49.932188
Dropout     32.120253
Enrolled    17.947559
Name: proportion, dtype: float64

In [189]:
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Duplicates:", df_clean.duplicated().sum())
print("Missing values:", df_clean.isna().sum().sum())

Original shape: (4424, 37)
Cleaned shape: (4424, 37)
Duplicates: 0
Missing values: 0


In [190]:
df_clean.head()

,marital_status,application_mode,application_order,course,daytime_evening_attendance,previous_qualification,previous_qualification_grade,nationality,mothers_qualification,fathers_qualification,...,curricular_units_2nd_sem_credited,curricular_units_2nd_sem_enrolled,curricular_units_2nd_sem_evaluations,curricular_units_2nd_sem_approved,curricular_units_2nd_sem_grade,curricular_units_2nd_sem_without_evaluations,unemployment_rate,inflation_rate,gdp,target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## previous's qualification column
### Labels represent:
1 - Secondary education 2 - Higher education - bachelor's degree 3 - Higher education - degree 4 - Higher education - master's 5 - Higher education - doctorate 6 - Frequency of higher education 9 - 12th year of schooling - not completed 10 - 11th year of schooling - not completed 12 - Other - 11th year of schooling 14 - 10th year of schooling 15 - 10th year of schooling - not completed 19 - Basic education 3rd cycle (9th/10th/11th year) or equiv. 38 - Basic education 2nd cycle (6th/7th/8th year) or equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 42 - Professional higher technical course 43 - Higher education - master (2nd cycle)

### segregating the labels based on the level of education - setting order

The codes are only labels, so we grouped them into levels.

Putting the codes into levels based on the highest education the student finished.


0 - Basic education - 38, 19 ;
1 - Secondary not finished - 15, 14, 10, 12, 9 ;
2 - Secondary finished - 1 ;
3 - After secondary, no degree - 39, 42, 6 ;
4 - Bachelor / degree - 2, 3, 40 ;
5 - Master - 4, 43 ;
6 - Doctorate - 5 ;

Used ordinal because education has an order.

Notes:
- 9, 10, 12, 14 and 15 are all 10th–12th year not completed, so I kept them together.
- 6 means they went to university but didn't finish, so I put it between secondary and degree.


In [191]:
prev_qual_level = {
    38: 0, 19: 0,
    15: 1, 14: 1, 10: 1, 12: 1, 9: 1,
    1: 2,
    39: 3, 42: 3, 6: 3,
    2: 4, 3: 4, 40: 4,
    4: 5, 43: 5,
    5: 6,
}
df_clean["prev_qual_level"] = df_clean["previous_qualification"].map(prev_qual_level)

In [192]:
df_clean["prev_qual_level"].value_counts()

prev_qual_level
2    3717
3     271
4     189
0     169
1      63
5      14
6       1
Name: count, dtype: int64

## Mother's qualification column:
### Labels represent
1 - Secondary Education - 12th Year of Schooling or Eq. 2 - Higher Education - Bachelor's Degree 3 - Higher Education - Degree 4 - Higher Education - Master's 5 - Higher Education - Doctorate 6 - Frequency of Higher Education 9 - 12th Year of Schooling - Not Completed 10 - 11th Year of Schooling - Not Completed 11 - 7th Year (Old) 12 - Other - 11th Year of Schooling 14 - 10th Year of Schooling 18 - General commerce course 19 - Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv. 22 - Technical-professional course 26 - 7th year of schooling 27 - 2nd cycle of the general high school course 29 - 9th Year of Schooling - Not Completed 30 - 8th year of schooling 34 - Unknown 35 - Can't read or write 36 - Can read without having a 4th year of schooling 37 - Basic education 1st cycle (4th/5th year) or equiv. 38 - Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 41 - Specialized higher studies course 42 - Professional higher technical course 43 - Higher Education - Master (2nd cycle) 44 - Higher Education - Doctorate (3rd cycle)	


### segregating the labels based on the level of education - setting order

Mother's qualification levels

0 - No schooling - 35, 36 ;
1 - Basic 1st cycle (4th year) - 37 ;
2 - Basic 2nd cycle (6th-8th year) - 38, 11, 26, 30, 29 ;
3 - Basic 3rd cycle (9th year) - 19, 27 ;
4 - Secondary not finished - 9, 10, 12, 14 ;
5 - Secondary finished - 1, 18, 22 ;
6 - After secondary, no degree - 6, 39, 42 ;
7 - Bachelor / degree - 2, 3, 40, 41 ;
8 - Master - 4, 43 ;
9 - Doctorate - 5, 44  ;
Unknown - 34 (left as NaN)  ;
    

In [193]:
mother_qual_level = {
    35: 0, 36: 0,
    37: 1,
    38: 2, 11: 2, 26: 2, 30: 2, 29: 2,
    19: 3, 27: 3,
    9: 4, 10: 4, 12: 4, 14: 4,
    1: 5, 18: 5, 22: 5,
    6: 6, 39: 6, 42: 6,
    2: 7, 3: 7, 40: 7, 41: 7,
    4: 8, 43: 8,
    5: 9, 44: 9,
}
df_clean["mothers_qualification"] = df_clean["mothers_qualification"].map(mother_qual_level)


In [194]:
(df_clean["mothers_qualification"] == 34).sum() # 130 unknowns

np.int64(0)

## Father's qualification column:
### Labels represent
1 - Secondary Education - 12th Year of Schooling or Eq. 2 - Higher Education - Bachelor's Degree 3 - Higher Education - Degree 4 - Higher Education - Master's 5 - Higher Education - Doctorate 6 - Frequency of Higher Education 9 - 12th Year of Schooling - Not Completed 10 - 11th Year of Schooling - Not Completed 11 - 7th Year (Old) 12 - Other - 11th Year of Schooling 13 - 2nd year complementary high school course 14 - 10th Year of Schooling 18 - General commerce course 19 - Basic Education 3rd Cycle (9th/10th/11th Year) or Equiv. 20 - Complementary High School Course 22 - Technical-professional course 25 - Complementary High School Course - not concluded 26 - 7th year of schooling 27 - 2nd cycle of the general high school course 29 - 9th Year of Schooling - Not Completed 30 - 8th year of schooling 31 - General Course of Administration and Commerce 33 - Supplementary Accounting and Administration 34 - Unknown 35 - Can't read or write 36 - Can read without having a 4th year of schooling 37 - Basic education 1st cycle (4th/5th year) or equiv. 38 - Basic Education 2nd Cycle (6th/7th/8th Year) or Equiv. 39 - Technological specialization course 40 - Higher education - degree (1st cycle) 41 - Specialized higher studies course 42 - Professional higher technical course 43 - Higher Education - Master (2nd cycle) 44 - Higher Education - Doctorate (3rd cycle)		no


### segregating the labels based on the level of education - setting order

Father's qualification levels

0 - No schooling - 35, 36 ;
1 - Basic 1st cycle (4th year) - 37 ;
2 - Basic 2nd cycle (6th-8th year) - 38, 11, 26, 30, 29 ;
3 - Basic 3rd cycle (9th year) - 19, 27 ;
4 - Secondary not finished - 9, 10, 12, 14, 25 ;
5 - Secondary finished - 1, 13, 18, 20, 22, 31, 33 ;
6 - After secondary, no degree - 6, 39, 42 ;
7 - Bachelor / degree - 2, 3, 40, 41 ;
8 - Master - 4, 43 ;
9 - Doctorate - 5, 44 ;
Unknown - 34 (left as NaN) ;

In [195]:
father_qual_level = {
    35: 0, 36: 0,
    37: 1,
    38: 2, 11: 2, 26: 2, 30: 2, 29: 2,
    19: 3, 27: 3,
    9: 4, 10: 4, 12: 4, 14: 4, 25: 4,
    1: 5, 13: 5, 18: 5, 20: 5, 22: 5, 31: 5, 33: 5,
    6: 6, 39: 6, 42: 6,
    2: 7, 3: 7, 40: 7, 41: 7,
    4: 8, 43: 8,
    5: 9, 44: 9,
}
df_clean["fathers_qualification"] = df_clean["fathers_qualification"].map(father_qual_level)

In [196]:
# df["Father's qualification"] .To check the unknown values

(df_clean["fathers_qualification"] == 34).sum() # 112 unknowns

np.int64(0)

## Mother's occupation column
### labels represent
0 - Student 1 - Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers 2 - Specialists in Intellectual and Scientific Activities 3 - Intermediate Level Technicians and Professions 4 - Administrative staff 5 - Personal Services, Security and Safety Workers and Sellers 6 - Farmers and Skilled Workers in Agriculture, Fisheries and Forestry 7 - Skilled Workers in Industry, Construction and Craftsmen 8 - Installation and Machine Operators and Assembly Workers 9 - Unskilled Workers 10 - Armed Forces Professions 90 - Other Situation 99 - (blank) 122 - Health professionals 123 - teachers 125 - Specialists in information and communication technologies (ICT) 131 - Intermediate level science and engineering technicians and professions 132 - Technicians and professionals, of intermediate level of health 134 - Intermediate level technicians from legal, social, sports, cultural and similar services 141 - Office workers, secretaries in general and data processing operators 143 - Data, accounting, statistical, financial services and registry-related operators 144 - Other administrative support staff 151 - personal service workers 152 - sellers 153 - Personal care workers and the like 171 - Skilled construction workers and the like, except electricians 173 - Skilled workers in printing, precision instrument manufacturing, jewelers, artisans and the like 175 - Workers in food processing, woodworking, clothing and other industries and crafts 191 - cleaning workers 192 - Unskilled workers in agriculture, animal production, fisheries and forestry 193 - Unskilled workers in extractive industry, construction, manufacturing and transport 194 - Meal preparation assistants

### Mother's occupation column is Nominal . There is no clear order among the occupations. 

The middle number in the lables are help to group them. example: 2, 122, 123, 125 are specialists

Mother's occupation groups

Student - 0 ;
Managers / directors - 1 ;
Specialists - 2, 122, 123, 125 ;
Technicians - 3, 131, 132, 134 ;
Administrative - 4, 141, 143, 144 ;
Services / sellers - 5, 151, 152, 153 ;
Farmers - 6  ;
Skilled industry / construction - 7, 171, 173, 175 ;
Machine operators - 8 ;
Unskilled workers - 9, 191, 192, 193, 194 ;
Armed forces - 10 ;
Other / unknown - 90, 99 ;


In [197]:
# Assigning the categorical names to the integer occupation labels

mother_occ_group = {
    0: "student",
    1: "managers",
    2: "specialists", 122: "specialists", 123: "specialists", 125: "specialists",
    3: "technicians", 131: "technicians", 132: "technicians", 134: "technicians",
    4: "administrative", 141: "administrative", 143: "administrative", 144: "administrative",
    5: "services_sellers", 151: "services_sellers", 152: "services_sellers", 153: "services_sellers",
    6: "farmers",
    7: "skilled_industry", 171: "skilled_industry", 173: "skilled_industry", 175: "skilled_industry",
    8: "machine_operators",
    9: "unskilled", 191: "unskilled", 192: "unskilled", 193: "unskilled", 194: "unskilled",
    10: "armed_forces",
    90: "other_unknown", 99: "other_unknown",
}
df_clean["mothers_occupation"] = df_clean["mothers_occupation"].map(mother_occ_group)

In [198]:
df_clean["mothers_occupation"].value_counts()

mothers_occupation
unskilled            1623
administrative        834
services_sellers      537
technicians           359
specialists           328
skilled_industry      279
student               144
managers              102
farmers                91
other_unknown          87
machine_operators      36
armed_forces            4
Name: count, dtype: int64

## Father's occupation column
### labels represent
0 - Student 1 - Representatives of the Legislative Power and Executive Bodies, Directors, Directors and Executive Managers 2 - Specialists in Intellectual and Scientific Activities 3 - Intermediate Level Technicians and Professions 4 - Administrative staff 5 - Personal Services, Security and Safety Workers and Sellers 6 - Farmers and Skilled Workers in Agriculture, Fisheries and Forestry 7 - Skilled Workers in Industry, Construction and Craftsmen 8 - Installation and Machine Operators and Assembly Workers 9 - Unskilled Workers 10 - Armed Forces Professions 90 - Other Situation 99 - (blank) 101 - Armed Forces Officers 102 - Armed Forces Sergeants 103 - Other Armed Forces personnel 112 - Directors of administrative and commercial services 114 - Hotel, catering, trade and other services directors 121 - Specialists in the physical sciences, mathematics, engineering and related techniques 122 - Health professionals 123 - teachers 124 - Specialists in finance, accounting, administrative organization, public and commercial relations 131 - Intermediate level science and engineering technicians and professions 132 - Technicians and professionals, of intermediate level of health 134 - Intermediate level technicians from legal, social, sports, cultural and similar services 135 - Information and communication technology technicians 141 - Office workers, secretaries in general and data processing operators 143 - Data, accounting, statistical, financial services and registry-related operators 144 - Other administrative support staff 151 - personal service workers 152 - sellers 153 - Personal care workers and the like 154 - Protection and security services personnel 161 - Market-oriented farmers and skilled agricultural and animal production workers 163 - Farmers, livestock keepers, fishermen, hunters and gatherers, subsistence 171 - Skilled construction workers and the like, except electricians 172 - Skilled workers in metallurgy, metalworking and similar 174 - Skilled workers in electricity and electronics 175 - Workers in food processing, woodworking, clothing and other industries and crafts 181 - Fixed plant and machine operators 182 - assembly workers 183 - Vehicle drivers and mobile equipment operators 192 - Unskilled workers in agriculture, animal production, fisheries and forestry 193 - Unskilled workers in extractive industry, construction, manufacturing and transport 194 - Meal preparation assistants 195 - Street vendors (except food) and street service providers

### Father's occupation column is Nominal . There is no clear order among the occupations. 

The middle number in the lables are help to group them. example: 5, 151, 152, 153, 154 are Services / sellers

Father's occupation groups

Student - 0 ;
Managers / directors - 1, 112, 114 ;
Specialists - 2, 121, 122, 123, 124 ;
Technicians - 3, 131, 132, 134, 135 ;
Administrative - 4, 141, 143, 144 ;
Services / sellers - 5, 151, 152, 153, 154 ;
Farmers - 6, 161, 163 ;
Skilled industry / construction - 7, 171, 172, 174, 175 ;
Machine operators - 8, 181, 182, 183 ;
Unskilled workers - 9, 192, 193, 194, 195 ;
Armed forces - 10, 101, 102, 103 ;
Other / unknown - 90, 99 ;


In [199]:
# Assigning the categorical names to the integer occupation labels

father_occ_group = {
    0: "student",
    1: "managers", 112: "managers", 114: "managers",
    2: "specialists", 121: "specialists", 122: "specialists", 123: "specialists", 124: "specialists",
    3: "technicians", 131: "technicians", 132: "technicians", 134: "technicians", 135: "technicians",
    4: "administrative", 141: "administrative", 143: "administrative", 144: "administrative",
    5: "services_sellers", 151: "services_sellers", 152: "services_sellers", 153: "services_sellers", 154: "services_sellers",
    6: "farmers", 161: "farmers", 163: "farmers",
    7: "skilled_industry", 171: "skilled_industry", 172: "skilled_industry", 174: "skilled_industry", 175: "skilled_industry",
    8: "machine_operators", 181: "machine_operators", 182: "machine_operators", 183: "machine_operators",
    9: "unskilled", 192: "unskilled", 193: "unskilled", 194: "unskilled", 195: "unskilled",
    10: "armed_forces", 101: "armed_forces", 102: "armed_forces", 103: "armed_forces",
    90: "other_unknown", 99: "other_unknown",
}
df_clean["fathers_occupation"] = df_clean["fathers_occupation"].map(father_occ_group)


In [200]:
df_clean["fathers_occupation"].value_counts()

fathers_occupation
unskilled            1034
skilled_industry      681
services_sellers      523
administrative        396
technicians           390
machine_operators     326
armed_forces          273
farmers               248
specialists           204
managers              137
student               128
other_unknown          84
Name: count, dtype: int64

## Admission grade column

In [201]:
df_clean["admission_grade"].describe()

count    4424.000000
mean      126.978119
std        14.482001
min        95.000000
25%       117.900000
50%       126.100000
75%       134.800000
max       190.000000
Name: admission_grade, dtype: float64

## Displaced column

In [202]:
df_clean["displaced"].value_counts()

displaced
1    2426
0    1998
Name: count, dtype: int64

In [203]:
df_clean.groupby("displaced")["target"].value_counts(normalize=True)
# Displaced students drop out less and also graduate more.

displaced  target  
0          Graduate    0.442943
           Dropout     0.376376
           Enrolled    0.180681
1          Graduate    0.545754
           Dropout     0.275763
           Enrolled    0.178483
Name: proportion, dtype: float64

## Educational special needs column

In [204]:
df_clean["educational_special_needs"].value_counts()

educational_special_needs
0    4373
1      51
Name: count, dtype: int64

In [205]:
df_clean.groupby("educational_special_needs")["target"].value_counts(normalize=True)
# Students who have educational special needs have less impact on the Drop out

educational_special_needs  target  
0                          Graduate    0.499886
                           Dropout     0.321061
                           Enrolled    0.179053
1                          Graduate    0.450980
                           Dropout     0.333333
                           Enrolled    0.215686
Name: proportion, dtype: float64

## Debtor column

In [206]:
df_clean["debtor"].value_counts(normalize=True)

debtor
0    0.886302
1    0.113698
Name: proportion, dtype: float64

In [207]:
df_clean.groupby("debtor")["target"].value_counts(normalize=True)
# Students who owe money to the institution are likely to drop out more

debtor  target  
0       Graduate    0.537618
        Dropout     0.282836
        Enrolled    0.179546
1       Dropout     0.620278
        Graduate    0.200795
        Enrolled    0.178926
Name: proportion, dtype: float64

## Tuition fees up to date column

In [208]:
df_clean["tuition_fees_up_to_date"].value_counts(normalize=True)

tuition_fees_up_to_date
1    0.880651
0    0.119349
Name: proportion, dtype: float64

In [209]:
df_clean.groupby("tuition_fees_up_to_date")["target"].value_counts(normalize=True)
# Students who stop paying fees are often students who have already left or are about to leave

tuition_fees_up_to_date  target  
0                        Dropout     0.865530
                         Enrolled    0.079545
                         Graduate    0.054924
1                        Graduate    0.559548
                         Dropout     0.247433
                         Enrolled    0.193018
Name: proportion, dtype: float64

## Gender column (1 = male, 0 = female)

In [210]:
df_clean["gender"].value_counts(normalize=True)

gender
0    0.648282
1    0.351718
Name: proportion, dtype: float64

In [211]:
df_clean.groupby("gender")["target"].value_counts(normalize=True)
#Men drop out more, Women graduate more.

gender  target  
0       Graduate    0.579149
        Dropout     0.251046
        Enrolled    0.169805
1       Dropout     0.450514
        Graduate    0.352185
        Enrolled    0.197301
Name: proportion, dtype: float64

## Cleaning and clustering data from Student Dropout Columns Marital Status, 

In [23]:
# Create new application order with 9 being top preference
df_clean["application_reordered"] = (df_clean["application_order"] - 10).abs()
df_clean = df_clean.drop(['application_order'], axis=1)

In [24]:
df_clean

,marital_status,application_mode,course,daytime_evening_attendance,previous_qualification,previous_qualification_grade,nationality,mothers_qualification,fathers_qualification,mothers_occupation,...,curricular_units_2nd_sem_enrolled,curricular_units_2nd_sem_evaluations,curricular_units_2nd_sem_approved,curricular_units_2nd_sem_grade,curricular_units_2nd_sem_without_evaluations,unemployment_rate,inflation_rate,gdp,target,application_reordered
0,1,17,171,1,1,122.0,1,19,12,5,...,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout,5
1,1,15,9254,1,1,160.0,1,1,3,3,...,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate,9
2,1,1,9070,1,1,122.0,1,37,37,9,...,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout,5
3,1,17,9773,1,1,122.0,1,38,37,5,...,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate,8
4,2,39,8014,0,1,100.0,1,37,38,9,...,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,1,1,9773,1,1,125.0,1,1,1,5,...,6,8,5,12.666667,0,15.5,2.8,-4.06,Graduate,4
4420,1,1,9773,1,1,120.0,105,1,1,9,...,6,6,2,11.000000,0,11.1,0.6,2.02,Dropout,8
4421,1,1,9500,1,1,154.0,1,37,37,9,...,8,9,1,13.500000,0,13.9,-0.3,0.79,Dropout,9
4422,1,1,9147,1,1,180.0,1,37,37,7,...,5,6,5,12.000000,0,9.4,-0.8,-3.12,Graduate,9


In [25]:
df_clean["application_reordered"].unique()

array([ 5,  9,  8,  6,  7,  4,  1, 10])

In [26]:
df_clean["application_mode"].value_counts()

application_mode
1     1708
17     872
39     785
43     312
44     213
7      139
18     124
42      77
51      59
16      38
53      35
15      30
5       16
10      10
2        3
57       1
26       1
27       1
Name: count, dtype: int64

In [27]:
mode_groups = {
    "1st phase special (islands)": [5, 16],
    "Transfer / change": [42, 43, 51, 57],
    "Prior qualification": [7, 44, 53],
    "Ordinances": [2, 10, 26, 27],
    "1st phase - general contingent": [1],
    "2nd phase - general contingent": [17],
    "Over 23 years old": [39],
    "3rd phase - general contingent": [18],
    "International student (bachelor)": [15],
}

code_to_group = {code: group for group, codes in mode_groups.items() for code in codes}
df_clean["application_mode_group"] = df_clean["application_mode"].replace(code_to_group).astype(str)

In [28]:
print(df_clean["application_mode_group"].isna().sum())   # should be 0
df_clean["application_mode_group"].value_counts()

0


application_mode_group
1st phase - general contingent      1708
2nd phase - general contingent       872
Over 23 years old                    785
Transfer / change                    449
Prior qualification                  387
3rd phase - general contingent       124
1st phase special (islands)           54
International student (bachelor)      30
Ordinances                            15
Name: count, dtype: int64

In [29]:
marriage_groups = {
    "single": [1],
    "married": [2],
    "widower": [3],
    "divorced": [4],
    "facto union": [5],
    "legally separated": [6],
}

code_to_marriage = {code: group for group, codes in marriage_groups.items() for code in codes}
df_clean["relationship_group"] = df_clean["marital_status"].replace(code_to_marriage).astype(str)

# Check: every code should have been converted, so no numbers should be left
print(df_clean["relationship_group"].value_counts())

relationship_group
single               3919
married               379
divorced               91
facto union            25
legally separated       6
widower                 4
Name: count, dtype: int64


In [30]:
marriage_groups = {
    "single": [1],
    "married": [2],
    "widower": [3],
    "divorced": [4],
    "facto union": [5],
    "legally separated": [6],
}

code_to_marriage = {code: group for group, codes in marriage_groups.items() for code in codes}
df_clean["relationship_group"] = df_clean["marital_status"].replace(code_to_marriage).astype(str)

# Check: every code should have been converted, so no numbers should be left
print(df_clean["relationship_group"].value_counts())

relationship_group
single               3919
married               379
divorced               91
facto union            25
legally separated       6
widower                 4
Name: count, dtype: int64


In [31]:
course_groups = {
    "Biofuel Production Technologies": [33],
    "Animation and Multimedia Design": [171],
    "Social Service (evening attendance)": [8014],
    "Agronomy": [9003],
    "Communication Design": [9070],
    "Veterinary Nursing": [9085],
    "Informatics Engineering": [9119],
    "Equinculture": [9130],
    "Management": [9147],
    "Social Service": [9238],
    "Tourism": [9254],
    "Nursing": [9500],
    "Oral Hygiene": [9556],
    "Advertising and Marketing Management": [9670],
    "Journalism and Communication": [9773],
    "Basic Education": [9853],
    "Management (evening attendance)": [9991],
}

code_to_course = {code: group for group, codes in course_groups.items() for code in codes}
df_clean["course_name"] = df_clean["course"].replace(code_to_course).astype(str)

In [32]:
qualification_groups = {
    "Secondary education": [1],
    "Secondary incomplete": [9, 10, 12, 14, 15],
    "Basic education": [19, 38],
    "Technical / vocational": [39, 42],
    "Frequency of higher education": [6],
    "Bachelor-level degree": [2, 3, 40],
    "Postgraduate": [4, 5, 43],
}

code_to_qualification = {code: group for group, codes in qualification_groups.items() for code in codes}
df_clean["previous_qualification_group"] = (
    df_clean["previous_qualification"].replace(code_to_qualification).astype(str)
)

# Check: every code should have been converted, so no numbers should be left
print(df_clean["previous_qualification_group"].value_counts())

previous_qualification_group
Secondary education              3717
Technical / vocational            255
Bachelor-level degree             189
Basic education                   169
Secondary incomplete               63
Frequency of higher education      16
Postgraduate                       15
Name: count, dtype: int64


In [33]:
df_clean["nationality"].value_counts()

nationality
1      4314
41       38
26       14
6        13
22       13
24        5
103       3
11        3
100       3
62        2
25        2
21        2
101       2
2         2
105       2
13        1
32        1
109       1
108       1
14        1
17        1
Name: count, dtype: int64

In [50]:
nationality_groups = {
    "Portuguese": [1],
    "Brazilian": [41],
    "Portuguese-speaking Africa": [21, 22, 24, 25, 26],
    "Other European": [2, 6, 11, 13, 14, 17, 62],
    "Eastern Europe and Turkey": [32, 100, 103, 105],
    "Latin America": [101, 108, 109],
}

code_to_nationality = {code: group for group, codes in nationality_groups.items() for code in codes}
df_clean["nationality_group"] = df_clean["nationality"].replace(code_to_nationality).astype(str)

In [51]:
df_clean["nationality_group"].value_counts()

nationality_group
Portuguese                    4314
Brazilian                       38
Portuguese-speaking Africa      36
Other European                  23
Eastern Europe and Turkey        9
Latin America                    4
Name: count, dtype: int64

In [52]:
df_clean

,marital_status,application_mode,course,daytime_evening_attendance,previous_qualification,previous_qualification_grade,nationality,mothers_qualification,fathers_qualification,mothers_occupation,...,unemployment_rate,inflation_rate,gdp,target,application_reordered,application_mode_group,relationship_group,course_name,previous_qualification_group,nationality_group
0,1,17,171,1,1,122.0,1,19,12,5,...,10.8,1.4,1.74,Dropout,5,2nd phase - general contingent,single,Animation and Multimedia Design,Secondary education,Portuguese
1,1,15,9254,1,1,160.0,1,1,3,3,...,13.9,-0.3,0.79,Graduate,9,International student (bachelor),single,Tourism,Secondary education,Portuguese
2,1,1,9070,1,1,122.0,1,37,37,9,...,10.8,1.4,1.74,Dropout,5,1st phase - general contingent,single,Communication Design,Secondary education,Portuguese
3,1,17,9773,1,1,122.0,1,38,37,5,...,9.4,-0.8,-3.12,Graduate,8,2nd phase - general contingent,single,Journalism and Communication,Secondary education,Portuguese
4,2,39,8014,0,1,100.0,1,37,38,9,...,13.9,-0.3,0.79,Graduate,9,Over 23 years old,married,Social Service (evening attendance),Secondary education,Portuguese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,1,1,9773,1,1,125.0,1,1,1,5,...,15.5,2.8,-4.06,Graduate,4,1st phase - general contingent,single,Journalism and Communication,Secondary education,Portuguese
4420,1,1,9773,1,1,120.0,105,1,1,9,...,11.1,0.6,2.02,Dropout,8,1st phase - general contingent,single,Journalism and Communication,Secondary education,Eastern Europe and Turkey
4421,1,1,9500,1,1,154.0,1,37,37,9,...,13.9,-0.3,0.79,Dropout,9,1st phase - general contingent,single,Nursing,Secondary education,Portuguese
4422,1,1,9147,1,1,180.0,1,37,37,7,...,9.4,-0.8,-3.12,Graduate,9,1st phase - general contingent,single,Management,Secondary education,Portuguese


In [56]:
df_clean_cat =df_clean.drop(['marital_status', 'application_mode', 'course', 'previous_qualification', 'nationality'], axis=1)
df_clean_cat

,daytime_evening_attendance,previous_qualification_grade,mothers_qualification,fathers_qualification,mothers_occupation,fathers_occupation,admission_grade,displaced,educational_special_needs,debtor,...,unemployment_rate,inflation_rate,gdp,target,application_reordered,application_mode_group,relationship_group,course_name,previous_qualification_group,nationality_group
0,1,122.0,19,12,5,9,127.3,1,0,0,...,10.8,1.4,1.74,Dropout,5,2nd phase - general contingent,single,Animation and Multimedia Design,Secondary education,Portuguese
1,1,160.0,1,3,3,3,142.5,1,0,0,...,13.9,-0.3,0.79,Graduate,9,International student (bachelor),single,Tourism,Secondary education,Portuguese
2,1,122.0,37,37,9,9,124.8,1,0,0,...,10.8,1.4,1.74,Dropout,5,1st phase - general contingent,single,Communication Design,Secondary education,Portuguese
3,1,122.0,38,37,5,3,119.6,1,0,0,...,9.4,-0.8,-3.12,Graduate,8,2nd phase - general contingent,single,Journalism and Communication,Secondary education,Portuguese
4,0,100.0,37,38,9,9,141.5,0,0,0,...,13.9,-0.3,0.79,Graduate,9,Over 23 years old,married,Social Service (evening attendance),Secondary education,Portuguese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4419,1,125.0,1,1,5,4,122.2,0,0,0,...,15.5,2.8,-4.06,Graduate,4,1st phase - general contingent,single,Journalism and Communication,Secondary education,Portuguese
4420,1,120.0,1,1,9,9,119.0,1,0,1,...,11.1,0.6,2.02,Dropout,8,1st phase - general contingent,single,Journalism and Communication,Secondary education,Eastern Europe and Turkey
4421,1,154.0,37,37,9,9,149.5,1,0,0,...,13.9,-0.3,0.79,Dropout,9,1st phase - general contingent,single,Nursing,Secondary education,Portuguese
4422,1,180.0,37,37,7,4,153.8,1,0,0,...,9.4,-0.8,-3.12,Graduate,9,1st phase - general contingent,single,Management,Secondary education,Portuguese
